In [ ]:
from grape import Graph, GraphVisualizer,EmbeddingResult
from IPython.display import display, HTML
import pandas as pd
import matplotlib.pyplot as plt
import os.path
# enable interactive dataframes
from itables import init_notebook_mode
init_notebook_mode(all_interactive=True)
plt.style.use('tableau-colorblind10')

In [ ]:
import os
# change working directory so that it picks up the grapehelper library
os.chdir('/home/ftorgano/rna-kg-analysis')
print(os.getcwd())

In [ ]:
from helper_lib import graph
import importlib
importlib.reload(graph)

In [ ]:
undirected_rnakg = graph.load_rnakg()
directed_rnakg = graph.load_rnakg(directed=True)

In [ ]:
from helper_lib import cache
import importlib
importlib.reload(cache)
cache.set_embedding_cache_dir("./RNA-KG_notebooks/Default_RNA-KG/cache/embeddings/")

In [ ]:
from grape.embedders import Node2VecSkipGramEnsmallen
n2v_sg_embedder = Node2VecSkipGramEnsmallen(enable_cache=False)
n2v_sg_dfs = cache.load_embeddings('Node2VecSkipGramEnsmallen_DFS')
print(type(n2v_sg_dfs))

In [ ]:
all_types = [undirected_rnakg.get_node_type_name_from_node_type_id(node_id) for node_id in range(undirected_rnakg.get_number_of_node_types())]
print(f'All types({len(all_types)}): {all_types}')
RNA_types = [t for t in all_types if 'RNA' in t]
print(f'RNA-related types({len(RNA_types)}): {RNA_types}')

In [ ]:
from helper_lib import visualize
import importlib
importlib.reload(visualize)

fig_size_latex = (9, 4)

visualize.plot_embedding(
  [n2v_sg_dfs[0]], 
  ["N2V SG DFS (0)", "N2V SG DFS (1)"], 
  [undirected_rnakg, undirected_rnakg],
  visualization_type='node_types',
  save_path='./RNA-KG_notebooks/Default_RNA-KG/reports/report3fixed/plots_functions/all_node_types_embedding',
  figsize=(fig_size_latex[1],fig_size_latex[1]),
  show=True,
  types_to_show=all_types,
  show_legend=True,
  number_of_subsampled_nodes=20_000,
  to_csv='./RNA-KG_notebooks/Default_RNA-KG/reports/report3fixed/plots_functions/tsne_embedding_node_types.csv',
)

In [ ]:
from helper_lib import visualize
import importlib
importlib.reload(visualize)

fig_size_latex = (9, 4)

visualize.plot_embedding(
  [n2v_sg_dfs[0]], 
  [""], 
  [undirected_rnakg, undirected_rnakg],
  visualization_type='node_types',
  save_path='./RNA-KG_notebooks/Default_RNA-KG/reports/report3fixed/plots_functions/RNA_node_types_embedding',
  figsize=(fig_size_latex[1],fig_size_latex[1]),
  show=True,
  types_to_show=RNA_types,
  k=10,
  show_legend=True,
  number_of_subsampled_nodes=20_000,
)

In [ ]:
from helper_lib import visualize
import importlib
importlib.reload(visualize)

fig_size_latex = (9, 4)

visualize.plot_embedding(
  [n2v_sg_dfs[0]], 
  [""], 
  [undirected_rnakg, undirected_rnakg],
  visualization_type='node_types',
  save_path='./RNA-KG_notebooks/Default_RNA-KG/reports/report3fixed/plots_functions/RNA_node_types_embedding_color_overlap',
  figsize=(fig_size_latex[1],fig_size_latex[1]),
  show=True,
  types_to_show=RNA_types,
  k=10,
  show_legend=True,
  number_of_subsampled_nodes=20_000,
)

In [ ]:
all_edge_types = [undirected_rnakg.get_edge_type_name_from_edge_type_id(edge_id) for edge_id in range(undirected_rnakg.get_number_of_edge_types())]
print(f'All edge types({len(all_edge_types)}): {all_edge_types}')
RNA_edge_types = [t for t in all_edge_types if 'RNA' in t]
print(f'RNA-related edge types({len(RNA_edge_types)}): {RNA_edge_types}')

In [ ]:
from helper_lib import visualize
import importlib
importlib.reload(visualize)

fig_size_latex = (9, 4)

visualize.plot_embedding(
  [n2v_sg_dfs[0]], 
  [""], 
  [undirected_rnakg, undirected_rnakg],
  visualization_type='edge_types',
  save_path='./RNA-KG_notebooks/Default_RNA-KG/reports/report3fixed/plots_functions/edge_types_miRNA_interacts__mRNA_regualtes_embedding',
  figsize=(fig_size_latex[1],fig_size_latex[1]),
  show=True,
  show_legend=True,
  number_of_subsampled_edges=20_000,
  to_csv='./RNA-KG_notebooks/Default_RNA-KG/reports/report3fixed/plots_functions/edge_types_miRNA_interacts__mRNA_regualtes_embedding.csv',
  triples_filter = [('miRNA','Interacts with', None),('mRNA','Regulates activity of', None)]
)

In [ ]:
from helper_lib import visualize
import importlib
importlib.reload(visualize)

fig_size_latex = (9, 4)

visualize.plot_embedding(
  [n2v_sg_dfs[0]], 
  ["miRNA interacts with and mRNA regulates activity of"],
  [undirected_rnakg, undirected_rnakg],
  visualization_type='edge_types',
  figsize=(fig_size_latex[1],fig_size_latex[1]),
  show=True,
  show_legend=True,
  number_of_subsampled_edges=20_000,
)

In [ ]:
import numpy as np
def get_flatten_unknown_edge_types(graph) -> np.ndarray:
    """Returns flattened edge type IDs adjusted for the current instance."""
    # The following is needed to normalize the unknown types
    unknown_edge_types_id = graph.get_number_of_edge_types()
    # When we have multiple node types for a given node, we set it to
    # the most common node type of the set.
    return np.fromiter(
        (
            unknown_edge_types_id if edge_type_id is None else edge_type_id
            for edge_type_id in (
                graph.get_directed_edge_type_ids()
                if graph.is_directed()
                else graph.get_upper_triangular_edge_type_ids()
            )
        ),
        dtype=np.uint32,
    )

In [ ]:
from helper_lib import visualize
import importlib
importlib.reload(visualize)

df = pd.DataFrame()
subsampled_graph = undirected_rnakg.sample_positive_graph(20_000, random_state=42)
if subsampled_graph.is_directed():
    edge_node_ids = (
      subsampled_graph.get_directed_source_node_ids(),
      subsampled_graph.get_directed_destination_node_ids(),
    )
else:
  edge_node_ids = (
    subsampled_graph.get_source_node_ids(directed=False),
    subsampled_graph.get_destination_node_ids(directed=False),
  )
sources = edge_node_ids[0]
destinations = edge_node_ids[1]
sources_types = [subsampled_graph.get_node_type_ids_from_node_id(node_id)[0] for node_id in sources]
sources_types_labels = [subsampled_graph.get_node_type_name_from_node_type_id(node_type_id) for node_type_id in sources_types]
destinations_types = [subsampled_graph.get_node_type_ids_from_node_id(node_id)[0] for node_id in destinations]
destinations_types_labels = [subsampled_graph.get_node_type_name_from_node_type_id(node_type_id) for node_type_id in destinations_types]
edge_types = get_flatten_unknown_edge_types(subsampled_graph)
edge_types_labels = [subsampled_graph.get_edge_type_name_from_edge_type_id(edge_type_id) for edge_type_id in edge_types]
df['source'] = sources
df['destination'] = destinations
df['source_type'] = sources_types
df['edge_type'] = edge_types
df['destination_type'] = destinations_types
df['source_type_label'] = sources_types_labels
df['edge_type_label'] = edge_types_labels
df['destination_type_label'] = destinations_types_labels
print(f"Dataframe len: {len(df)}")

triples_filter = [('Disease','Causes or contributes to condition', 'miRNA')]

for (source_type, edge_type, destination_type) in triples_filter:
  print(f"Filtering triples with source type: {source_type}, edge type: {edge_type}, destination type: {destination_type}")
  # convert types to ids
  source_type_id = subsampled_graph.get_node_type_id_from_node_type_name(source_type) if source_type is not None else None
  edge_type_id = subsampled_graph.get_edge_type_id_from_edge_type_name(edge_type) if edge_type is not None else None
  destination_type_id = subsampled_graph.get_node_type_id_from_node_type_name(destination_type) if destination_type is not None else None
  source_type_id_filter = df['source_type']==source_type_id if source_type_id is not None else True
  edge_type_id_filter = df['edge_type']==edge_type_id if edge_type_id is not None else True
  destination_type_id_filter = df['destination_type']==destination_type_id if destination_type_id is not None else True
  df = df[source_type_id_filter & edge_type_id_filter & destination_type_id_filter]

print(f"Dataframe len: {len(df)}")
df

In [ ]:
from helper_lib import visualize
import importlib
importlib.reload(visualize)

fig_size_latex = (9.000000000000002, 4)

visualize.plot_embedding(
  [n2v_sg_dfs[0]], 
  [""], 
  [undirected_rnakg, undirected_rnakg],
  visualization_type='triples_types',
  triples_filter = [('miRNA',None, 'mRNA')],
  save_path='./RNA-KG_notebooks/Default_RNA-KG/reports/report3fixed/plots_functions/triples',
  figsize=(fig_size_latex[1],fig_size_latex[1]),
  show=True,
  show_legend=True,
  number_of_subsampled_edges=20_000,
  legend_cols=2
)